# NB06: 공공시설 (유휴공간 후보) 좌표화 및 통합

**목적**: 드론 스테이션 후보 부지로 활용 가능한 공공시설물을 GeoDataFrame으로 통합

**입력**:
- 주차장 정보 CSV (`서울시 공영주차장 안내 정보.csv`)

**출력**: `processed/public_facilities.gpkg`

In [1]:
import json
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

BASE = Path("..").resolve()
RAW  = BASE / "00_data"
OUT  = BASE / "processed"
OUT.mkdir(exist_ok=True)

## 1. 주차장 데이터 로드 및 전처리

In [2]:
parking_json_path = RAW / "서울시 공영주차장 안내 정보.json"
if not parking_json_path.exists():
    raise FileNotFoundError(parking_json_path)

with open(parking_json_path, encoding="utf-8") as f:
    raw = json.load(f)

records = raw.get("DATA", raw) if isinstance(raw, dict) else raw
pk_df = pd.DataFrame(records)

# Normalize column names to lowercase
pk_df.columns = pk_df.columns.str.lower()

# Map JSON fields to output columns
# lat=latitude, lot=longitude, tpkct=total capacity
pk_df["lat"] = pd.to_numeric(pk_df["lat"], errors="coerce")
pk_df["lon"] = pd.to_numeric(pk_df["lot"], errors="coerce")
pk_df["capacity"] = pd.to_numeric(pk_df["tpkct"], errors="coerce").fillna(0).astype(int)

# Drop rows with missing coordinates
pk_valid = pk_df.dropna(subset=["lat", "lon"]).copy()
print(f"주차장 원본: {len(pk_df)}개소")
print(f"좌표 정상: {len(pk_valid)}개소")

gdf_parking = gpd.GeoDataFrame(
    pk_valid,
    geometry=gpd.points_from_xy(pk_valid["lon"], pk_valid["lat"]),
    crs="EPSG:4326",
)

gdf_parking = gdf_parking[["pklt_cd", "pklt_nm", "pklt_knd_nm", "oper_se_nm", "addr", "capacity", "lat", "lon", "geometry"]].rename(columns={
    "pklt_cd":     "id",
    "pklt_nm":     "name",
    "pklt_knd_nm": "type",
    "oper_se_nm":  "subtype",
    "addr":        "address",
})
gdf_parking["facility"] = "주차장"

scaler = MinMaxScaler()
gdf_parking["idle_score"] = scaler.fit_transform(gdf_parking[["capacity"]])

print(f"유효 좌표 확인: {gdf_parking.geometry.is_valid.sum()} / {len(gdf_parking)}")
print(gdf_parking[["id", "name", "capacity", "lat", "lon"]].head(3))

주차장 원본: 2292개소
좌표 정상: 1531개소
유효 좌표 확인: 1531 / 1531
         id               name  capacity        lat         lon
7   1037932  구로디지털단지역 공영주차장(시)        91  37.485432  126.901243
16  1051043      구파발역 공영주차장(시)       399  37.638209  126.918987
22  1163833     봉천복개3 공영주차장(시)         1  37.488068  126.931392


## 2. 통합 GeoDataFrame 저장

In [3]:
gdf_parking.to_file(OUT / "public_facilities.gpkg", driver="GPKG")
print(f"저장 완료: {OUT / 'public_facilities.gpkg'} ({len(gdf_parking)}개소)")

저장 완료: C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3\processed\public_facilities.gpkg (1531개소)


## 3. 지도 시각화

In [4]:
parking_records = [
    {"name": row["name"], "lat": row["lat"], "lon": row["lon"], "capacity": int(row["capacity"])}
    for _, row in gdf_parking.iterrows()
]
json_out = OUT / "public_parking.json"
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(parking_records, f, ensure_ascii=False, indent=2)
print(f"public_parking.json 저장 완료: {json_out} ({len(parking_records)}개소)")

public_parking.json 저장 완료: C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3\processed\public_parking.json (1531개소)


In [5]:
check = gpd.read_file(OUT / "public_facilities.gpkg")
assert len(check) > 0, "public_facilities.gpkg is empty"
assert all(c in check.columns for c in ["name", "facility", "capacity", "lat", "lon"]),     f"Missing columns: {[c for c in ['name','facility','capacity','lat','lon'] if c not in check.columns]}"

with open(OUT / "public_parking.json", encoding="utf-8") as f:
    chk_json = json.load(f)
assert len(chk_json) > 0, "public_parking.json is empty"
assert all(k in chk_json[0] for k in ["name", "lat", "lon"]), "public_parking.json missing required keys"

print("Validation passed")
print(f"  public_facilities.gpkg: {len(check)} rows")
print(f"  public_parking.json:    {len(chk_json)} entries")

Validation passed
  public_facilities.gpkg: 1531 rows
  public_parking.json:    1531 entries
